In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

def compare_groups_bootstrap_t(
    df,
    group1,
    group2,
    bag_metric="GAP_corrected_M1",
    metrics=None,
    diagnosis_col="clinical_diagnosis",
    age_col="demo_age",
    sex_col="demo_sex",
    education_col="cog_ed",
    n_reps=1000,
    seed1=0,
    seed2=1,
    valid_diagnoses=None,
    plot=True,
    remove_outliers=True,
    ax=None
):

    if metrics is None:
        metrics = [
            bag_metric,
            "concreteness-all",
            "granularity_extraction-all",
            "psycholinguistic_objective-all",
            "pitch_analysis-all",
            "sentiment_analysis-all",
            "talking_intervals-all",
            "verbosity-all"
        ]

    if valid_diagnoses is None:
        valid_diagnoses = ["CN", "AD", "FTD", "FTD-L", "DCL"]

    rng_1 = np.random.default_rng(seed1)
    rng_2 = np.random.default_rng(seed2)

    df_work = df.copy()

    # Convert numeric columns
    for col in metrics + [age_col, education_col]:
        if col in df_work.columns:
            df_work[col] = pd.to_numeric(df_work[col], errors="coerce")

    df_work[sex_col] = df_work[sex_col].astype(str).str.strip()
    df_work[diagnosis_col] = df_work[diagnosis_col].astype(str).str.strip()

    df_work = df_work[
        df_work[diagnosis_col].isin(valid_diagnoses)
    ].dropna(
        subset=[diagnosis_col, age_col, education_col, sex_col] + metrics
    ).copy()

    sex_dummies = pd.get_dummies(df_work[sex_col].astype(str), prefix="sex", drop_first=True)
    X_df = pd.concat([df_work[[age_col, education_col]].astype(float), sex_dummies], axis=1)
    X = np.column_stack([np.ones(len(df_work)), X_df.to_numpy(dtype=float)])

    metrics_adj = []
    for metric in metrics:
        y = df_work[metric].to_numpy(dtype=float)
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        adjusted_name = metric + "_adjdemo"
        df_work[adjusted_name] = y - (X @ beta)
        metrics_adj.append(adjusted_name)

    dsub = df_work[df_work[diagnosis_col].isin([group1, group2])].copy()

    res = pd.DataFrame(index=dsub.index)
    res["dx"] = dsub[diagnosis_col].values

    for metric_adj in metrics_adj:
        res[metric_adj] = dsub[metric_adj].to_numpy(dtype=float)

    idx_1 = np.where(res["dx"].to_numpy() == group1)[0]
    idx_2 = np.where(res["dx"].to_numpy() == group2)[0]

    n_1 = len(idx_1)
    n_2 = len(idx_2)

    if n_1 < 2 or n_2 < 2:
        raise ValueError(f"Not enough samples: {group1}={n_1}, {group2}={n_2} (need >=2 each).")

    tvals = np.zeros((n_reps, len(metrics_adj)), dtype=float)

    for rep in range(n_reps):
        sample_1 = rng_1.choice(idx_1, size=n_1, replace=True)
        sample_2 = rng_2.choice(idx_2, size=n_2, replace=True)

        for j, metric_adj in enumerate(metrics_adj):
            a = res.iloc[sample_1][metric_adj].to_numpy(dtype=float)
            b = res.iloc[sample_2][metric_adj].to_numpy(dtype=float)
            tvals[rep, j] = stats.ttest_ind(a, b, equal_var=False).statistic

        if (rep + 1) % 100 == 0 or (rep + 1) == n_reps:
            print(f"{group1} vs {group2}: {rep + 1}/{n_reps}")

    abs_tvals = np.abs(tvals)

    if plot:
        mean_abs_t = abs_tvals.mean(axis=0)
        sort_idx = np.argsort(mean_abs_t)[::-1]

        metrics_sorted = [metrics[i] for i in sort_idx]
        plot_data = [abs_tvals[:, i] for i in sort_idx]
        mean_abs_t_sorted = mean_abs_t[sort_idx]

        if ax is None:
            fig, ax = plt.subplots(figsize=(7, 5))

        ax.violinplot(
            plot_data,
            showmeans=True,
            showextrema=False,
            widths=0.9,
            vert=False
        )

        ax.set_title(f"{group1} vs {group2}")
        ax.set_yticks(np.arange(1, len(metrics_sorted) + 1))
        ax.set_yticklabels(
            [f"{metric} (mean|t|={value:.2f})" for metric, value in zip(metrics_sorted, mean_abs_t_sorted)]
        )
        ax.set_xlabel("|t| on residuals")
        ax.grid(True, axis="x", alpha=0.25)

    if remove_outliers:
        mean_vals = abs_tvals.mean(axis=0)
        std_vals = abs_tvals.std(axis=0, ddof=1)

        keep_mask = np.abs(abs_tvals - mean_vals) <= (2 * std_vals)
        keep_mask = np.sum(keep_mask, axis=1) == len(metrics_adj)

        abs_tvals = abs_tvals[keep_mask, :]

    bag_idx = metrics.index(bag_metric)
    bag_vals = abs_tvals[:, bag_idx]

    rows = []
    for j, metric in enumerate(metrics):
        if metric == bag_metric:
            continue

        metric_vals = abs_tvals[:, j]
        diff = bag_vals - metric_vals

        sd_diff = diff.std(ddof=1)
        dz = diff.mean() / sd_diff if sd_diff > 0 else np.nan
        ci_low, ci_high = np.percentile(diff, [2.5, 97.5])

        p_bootstrap = 2 * min(np.mean(diff <= 0), np.mean(diff >= 0))
        p_bootstrap = max(p_bootstrap, 1.0 / (len(diff) + 1))

        rows.append({
            "comparison": f"{group1} vs {group2}",
            "metric": metric,
            "mean_diff_abs_t": diff.mean(),
            "ci95_diff_abs_t_low": ci_low,
            "ci95_diff_abs_t_high": ci_high,
            "p_bootstrap": p_bootstrap,
            "cohen_dz_paired": dz
        })

    table = pd.DataFrame(rows)

    # FDR correction
    pvals_arr = table["p_bootstrap"].to_numpy(dtype=float)
    n_tests = pvals_arr.size
    order = np.argsort(pvals_arr)
    ranked = pvals_arr[order]

    qvals = ranked * n_tests / np.arange(1, n_tests + 1)
    qvals = np.minimum.accumulate(qvals[::-1])[::-1]
    qvals = np.clip(qvals, 0, 1)

    qvals_out = np.empty_like(qvals)
    qvals_out[order] = qvals

    table["q_bh_fdr"] = qvals_out
    table = table.sort_values("q_bh_fdr").reset_index(drop=True)

    return table, abs_tvals, df_work

## Load data

In [ ]:
df_SAGs = pd.read_excel('../../Data/SAG-prediction-power.xlsx')

In [ ]:

bag_metric = "GAP_corrected_M1"

metrics = [
    bag_metric,
    'Semantic-memory-comp',
     'Processing-speed-comp',
     'Affect-comp',
     'Linguistic-comp',
     'Acustic-comp'
]

comparisons = [
    ("AD", "CN"),
    ("FTD", "CN"),
    ("FTD-L", "CN")
]

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
plt.rcParams.update({"font.size": 12})

tables = []

for i, (g1, g2) in enumerate(comparisons):
    table_tmp, abs_tvals_tmp, df_resid_tmp = compare_groups_bootstrap_t(
        df=df_SAGs,
        group1=g1,
        group2=g2,
        bag_metric=bag_metric,
        metrics=metrics,
        diagnosis_col="clinical_diagnosis",
        age_col="demo_age",
        sex_col="demo_sex",
        education_col="cog_ed",
        n_reps=1000,
        seed1=0,
        seed2=1,
        valid_diagnoses=["CN", "AD", "FTD", "FTD-L", "DCL"],
        plot=True,
        remove_outliers=True,
        ax=axes[i]        
    )
    
    tables.append(table_tmp)

for ax in axes:
    ax.set_xlim(0, 30)


plt.tight_layout()

plt.show()

final_table = pd.concat(tables, ignore_index=True)


pvals = final_table['p_bootstrap'].values

rejected, pvals_fdr, _, _ = multipletests(pvals, method='fdr_bh')

final_table['p_fdr'] = pvals_fdr
final_table['significant_fdr'] = rejected

display(final_table[['comparison', 'metric', 'p_bootstrap', 'p_fdr', 'significant_fdr']])
